# 4 — API Reference

This notebook documents every public class and function in the
`chemical_reservoir` package with usage examples.

| Module | Key classes / functions |
|--------|------------------------|
| `reservoir` | `ConstrainedChemicalReservoir` |
| `problems` | `ODEProblem`, `PDEProblem1D`, factory functions |
| `constraints` | `PhysicalConstraints`, `MeasurementConstraints`, `SpeciesBounds` |
| `optimization` | `BayesianOptimizer`, acquisition functions |
| `metrics` | `LearningMetrics`, `MetricsCalculator` |
| `sensitivity` | `SensitivityAnalyzer` |
| `visualization` | `ResultsVisualizer`, `save_figure` |
| `network_viz` | `ReactionNetworkVisualizer` |
| `io` | `load_reservoir_from_json`, `save_reaction_network`, etc. |
| `networks` | `get_builtin_networks`, `list_builtin_networks` |
| `config` | `ChemCompConfig`, `load_config`, `get_config`, `update_config` |
| `validate` | `validate_problem_file`, `validate_network_file` |
| `custom_problems` | `load_problem_from_yaml` |

In [ ]:
import numpy as np
import chemical_reservoir as cr
from pathlib import Path

NETWORK_DIR = Path("../network_configs")

---
## 4.1  `ConstrainedChemicalReservoir`

The central class.  It wraps a chemical reaction network, simulates
mass-action kinetics, and trains a ridge-regression readout.

Two prediction modes are supported:
- **Open-loop** (`solve_ode`): the input encoding drives the reservoir over the full time domain; predictions are made in a single forward pass.
- **Closed-loop** (`solve_ode_closed_loop`): training uses teacher forcing; testing feeds predicted outputs back to modulate the encoding signal, enabling autonomous attractor tracking.

### Constructor

```python
ConstrainedChemicalReservoir(
    reactions: List[Dict],
    input_channels: List[str],
    output_channels: List[str],
    physical_constraints: PhysicalConstraints,
    measurement_constraints: MeasurementConstraints,
    feature_expansion: str = 'quadratic',  # 'none' or 'quadratic'
)
```

### Key methods

| Method | Description |
|--------|-------------|
| `set_initial_concentrations(concs)` | Set starting concentrations (dict) |
| `set_input_rates(rates)` | Set input rate scaling (dict) |
| `solve_ode(problem, n_train, n_test, ...)` | Solve an ODE problem (open-loop) |
| `solve_ode_closed_loop(problem, n_train, n_test, feedback_scale, ...)` | Solve an ODE with closed-loop output feedback |
| `solve_pde(problem, n_train_time, n_test_time, ...)` | Solve a 1-D PDE (open-loop) |
| `optimize_with_bayesian(problem, ...)` | Bayesian optimisation loop |
| `plot_reaction_network(...)` | Draw the network graph |
| `report()` | Print a summary of the reservoir |

In [ ]:
# Example: manual construction
reactions = [
    {"species": ["A", "B"], "reactants": [1, 1], "products": [2, 0], "rate": 1.0},
    {"species": ["B", "A"], "reactants": [1, 0], "products": [0, 1], "rate": 0.5},
    {"species": ["A", "B"], "reactants": [2, 0], "products": [1, 1], "rate": 0.1},
]

physical = cr.PhysicalConstraints(
    default_min_concentration=0.0,
    default_max_concentration=5.0,
    default_min_input_rate=0.0,
    default_max_input_rate=2.0,
)
physical.add_species_bounds("A", min_conc=0, max_conc=4, min_rate=0, max_rate=1.5)
physical.add_species_bounds("B", min_conc=0, max_conc=4, min_rate=0, max_rate=1.5)

measurement = cr.MeasurementConstraints(
    max_simultaneous_measurements=2,
    measurable_species={"A", "B"},
    unmeasurable_species=set(),
    measurement_noise_std=0.01,
)

reservoir = cr.ConstrainedChemicalReservoir(
    reactions=reactions,
    input_channels=["A"],
    output_channels=["A", "B"],
    physical_constraints=physical,
    measurement_constraints=measurement,
    feature_expansion="quadratic",
)

reservoir.set_initial_concentrations({"A": 1.0, "B": 1.0})
reservoir.set_input_rates({"A": 0.5})

reservoir.report()

---
## 4.2  `ODEProblem` & `PDEProblem1D`

### `ODEProblem`

```python
ODEProblem(
    func: Callable,       # dy/dt = func(y, t, *params)
    params: tuple,
    y0: array-like,
    t_span: (t_start, t_end),
    variable_names: List[str],
    name: str,
)
```

| Method | Description |
|--------|-------------|
| `solve_numerical(t_eval)` | Solve with `scipy.integrate.odeint` |
| `get_train_test_times(n_train, n_test, horizon)` | Generate time arrays |

### `PDEProblem1D`

```python
PDEProblem1D(
    pde_func: Callable,   # du/dt = f(u, x, t, u_x, u_xx, *params)
    params: tuple,
    x_domain: (x_min, x_max),
    t_span: (t_start, t_end),
    initial_condition: Callable,
    boundary_conditions: Dict,
    n_spatial: int,
    name: str,
)
```

### Factory functions

All in `chemical_reservoir.problems`:

In [ ]:
# List all factory functions
factories = [
    ("create_schnakenberg_problem", "ODE"),
    ("create_vanderpol_problem", "ODE"),
    ("create_brusselator_problem", "ODE"),
    ("create_lotka_volterra_problem", "ODE"),
    ("create_lorenz_problem", "ODE"),
    ("create_heat_equation_problem", "PDE"),
    ("create_advection_diffusion_problem", "PDE"),
    ("create_fisher_equation_problem", "PDE"),
    ("create_burgers_equation_problem", "PDE"),
]

for name, kind in factories:
    fn = getattr(cr.problems, name)
    prob = fn()
    if kind == "ODE":
        print(f"{kind}  {name:<40} → {prob.name} ({prob.variable_names})")
    else:
        print(f"{kind}  {name:<40} → {prob.name} (nx={prob.n_spatial})")

---
## 4.3  `PhysicalConstraints` & `MeasurementConstraints`

Constraints enforce physically meaningful reservoir states.

In [ ]:
# PhysicalConstraints — global + per-species bounds
pc = cr.PhysicalConstraints(
    default_min_concentration=0.0,
    default_max_concentration=10.0,
    default_min_input_rate=0.0,
    default_max_input_rate=3.0,
)
pc.add_species_bounds("A", min_conc=0, max_conc=5, min_rate=0, max_rate=2)
print(f"Bounds for A: {pc.get_bounds('A')}")
print(f"Bounds for unknown: {pc.get_bounds('Z')}")

In [ ]:
# MeasurementConstraints — which species can be observed
mc = cr.MeasurementConstraints(
    max_simultaneous_measurements=3,
    measurable_species={"A", "B", "C"},
    unmeasurable_species={"D"},
    measurement_noise_std=0.05,
)
print(f"Measurable: {mc.measurable_species}")
print(f"Noise std: {mc.measurement_noise_std}")

---
## 4.4  `MetricsCalculator` & `LearningMetrics`

### Static methods on `MetricsCalculator`

| Method | Returns |
|--------|---------|  
| `calculate_all(y_true, y_pred, t, n_train, ...)` | `LearningMetrics` dataclass |
| `calculate_pde_metrics(sol_true, sol_pred, x, t, n_train)` | dict |
| `rmse(y_true, y_pred)` | float |
| `mae(y_true, y_pred)` | float |
| `r_squared(y_true, y_pred)` | float |
| `pearson_correlation(y_true, y_pred)` | float |

In [ ]:
# Quick standalone metric calculation
y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = np.array([1.1, 2.2, 2.8, 4.1, 4.9])

rmse = np.sqrt(np.mean((y_true - y_pred)**2))
mae = np.mean(np.abs(y_true - y_pred))
ss_res = np.sum((y_true - y_pred)**2)
ss_tot = np.sum((y_true - y_true.mean())**2)
r2 = 1 - ss_res / ss_tot
pearson = np.corrcoef(y_true, y_pred)[0, 1]

print(f"RMSE:    {rmse:.4f}")
print(f"MAE:     {mae:.4f}")
print(f"R²:      {r2:.4f}")
print(f"Pearson: {pearson:.4f}")

# Or use calculate_all for a full metrics object (requires 2-D arrays)
y_true_2d = y_true.reshape(-1, 1)
y_pred_2d = y_pred.reshape(-1, 1)
t = np.linspace(0, 1, len(y_true))
full_metrics = cr.MetricsCalculator.calculate_all(
    y_true_2d, y_pred_2d, t, n_train=3, variable_names=["y"]
)
print(f"\nVia calculate_all → Train RMSE: {full_metrics.train_rmse:.4f}, "
      f"Test RMSE: {full_metrics.test_rmse:.4f}")

---
## 4.5  `BayesianOptimizer`

### Constructor

```python
BayesianOptimizer(
    bounds: Dict[str, Tuple[float, float]],
    acquisition_function: AcquisitionFunction = ExpectedImprovement(),
    n_initial: int = 10,
)
```

### Acquisition functions

- `ExpectedImprovement(xi=0.01)` — best for exploitation
- `ProbabilityOfImprovement(xi=0.01)` — most greedy
- `UpperConfidenceBound(kappa=2.0)` — best for exploration

In [ ]:
# Standalone optimiser example (2D Rosenbrock)
optimizer = cr.BayesianOptimizer(
    bounds_lower=np.array([-2.0, -2.0]),
    bounds_upper=np.array([2.0, 2.0]),
    variable_names=["x", "y"],
    acquisition="EI",
    n_initial=5,
)

def rosenbrock(params):
    x, y = params
    return (1 - x)**2 + 100 * (y - x**2)**2

# Run a few iterations
for i in range(15):
    next_point = optimizer.suggest()
    value = rosenbrock(next_point)
    optimizer.observe(next_point, value)

print(f"Best value found: {optimizer.best_y:.4f}")
print(f"Best parameters:  x={optimizer.best_x[0]:.4f}, y={optimizer.best_x[1]:.4f}")

---
## 4.6  `SensitivityAnalyzer`

```python
SensitivityAnalyzer(reservoir, problem)
```

| Method | What it probes |
|--------|----------------|
| `analyze_concentration_sensitivity(n_train)` | Global & per-species concentration |
| `analyze_input_rate_sensitivity(n_train)` | Global & per-channel input rates |
| `analyze_measurement_sensitivity(n_train)` | Different measurement subsets |
| `compute_local_sensitivity(n_train)` | Finite-difference Jacobian |

Results stored in `analyzer.sensitivity_results` (dict).

---
## 4.7  `ResultsVisualizer`

```python
ResultsVisualizer(config: OutputConfig = None)
```

| Method | Panels |
|--------|--------|
| `plot_learning_metrics_dashboard(results, metrics, problem)` | 10-subplot dashboard |
| `plot_pde_metrics(results, pde_metrics)` | 6-subplot PDE dashboard |
| `plot_optimization_analysis(opt_results)` | 6-subplot BO dashboard |
| `plot_sensitivity_analysis(sens_results)` | 8-subplot sensitivity dashboard |
| `plot_horizon_comparison(horizon_data)` | 4-subplot horizon study |

---
## 4.8  I/O Module (`chemical_reservoir.io`)

| Function | Description |
|----------|-------------|
| `load_reaction_network(filepath)` | Parse & validate JSON → dict |
| `save_reaction_network(filepath, ...)` | Write network dict → JSON |
| `load_reservoir_from_json(filepath, ...)` | JSON → `ConstrainedChemicalReservoir` |
| `load_all_networks(config_dir)` | Load all `*.json` in a directory |
| `validate_reaction(rxn, index)` | Validate a single reaction dict |

In [ ]:
# Load all shipped networks
all_configs = cr.io.load_all_networks(NETWORK_DIR)
for name, cfg in all_configs.items():
    print(f"{name:<25} {len(cfg['reactions'])} reactions, "
          f"{len(cfg['all_species'])} species")

---
## 4.9  `OutputConfig` & `set_output_directory`

Control where figures are saved and in what format.

```python
OutputConfig(
    output_dir: str = "results",
    figure_format: str = "pdf",   # 'pdf', 'png', 'svg'
    save_figures: bool = True,
    show_figures: bool = False,
)
```

In [ ]:
# Example: save all figures as PNG to a custom directory
config = cr.OutputConfig(
    output_dir="my_results",
    figure_format="png",
    save_figures=True,
    show_figures=True,
)

vis = cr.ResultsVisualizer(config=config)
print(f"Figures will be saved to: {config.output_dir}/")

---
## 4.10  Networks Module (`chemical_reservoir.networks`)

| Function | Description |
|----------|-------------|
| `get_builtin_networks()` | Returns dict of all built-in network reaction lists |
| `get_network_io_config(name)` | Default (input, output) channels for a network |
| `list_builtin_networks()` | List of network names |

In [ ]:
from chemical_reservoir.networks import get_builtin_networks, get_network_io_config

networks = get_builtin_networks()
for name in networks:
    inp, out = get_network_io_config(name)
    print(f"{name:<25} in={inp}  out={out}")

---
## 4.11  Validation (`chemical_reservoir.validate`)

Check that user-provided problem YAML or network JSON files are well-formed
before running a full solve.

| Function | Description |
|----------|-------------|
| `validate_problem_file(filepath)` | Validate a YAML problem definition → list of errors |
| `validate_network_file(filepath)` | Validate a JSON network definition → list of errors |
| `main()` | CLI entry point (also available as `chemcomp-validate`) |

Validation checks include: file parsing, required fields, SymPy equation
parsing, stoichiometry/IC length consistency, and a short integration smoke test.

In [ ]:
from chemical_reservoir.validate import validate_problem_file, validate_network_file

# Validate a correct problem file
errors = validate_problem_file("../problem_configs/vanderpol.yaml")
print(f"vanderpol.yaml: {'✓ Valid' if not errors else errors}")

# Validate a correct network file
errors = validate_network_file("../network_configs/cascade.json")
print(f"cascade.json:   {'✓ Valid' if not errors else errors}")

---
## Summary

This notebook covered every public API surface of the `chemical_reservoir`
package.  For the mathematical background see **Notebook 01**, for usage
tutorials see **Notebooks 02–03**, and for a comparison of open-loop vs
closed-loop prediction modes see **Notebook 05**.